# convtranspose-bn-activation-block composite — cx2: Rearrange unflattens the latent into a feature map before a ConvT/BN/ReLU block

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `convtranspose-bn-activation-block`, `rearrange-as-sequential-layer`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "convtranspose-bn-activation-block"
DD_ATOM_IDS = ["convtranspose-bn-activation-block", "rearrange-as-sequential-layer"]
DD_SUBTOPICS = ["GAN: ConvT+BN+Activation block", "Einops: Rearrange as nn.Sequential layer"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A DCGAN generator can start from a flat latent code `z` of shape `(B, C*H*W)` and lift it directly into a feature map ready for the first ConvT block. Two atoms wire this together inside a single `nn.Sequential`:

1. **rearrange-as-sequential-layer** — `einops.layers.torch.Rearrange` is the *module* form of `einops.rearrange`. It can sit inside `nn.Sequential` as a real layer, no `forward` lambda required. Common pattern: `Rearrange('b (c h w) -> b c h w', c=C, h=H, w=W)`.
2. **convtranspose-bn-activation-block** — `ConvTranspose2d(kernel=4, stride=2, padding=1, bias=False) -> BatchNorm2d -> ReLU`, the canonical generator upsampling unit (spatial *= 2, channels halve).

**Anatomy.**
```python
nn.Sequential(
    Rearrange('b (c h w) -> b c h w', c=128, h=4, w=4),       # rearrange-as-sequential-layer.
    nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),         # convtranspose-bn-activation-block.
    nn.BatchNorm2d(64),
    nn.ReLU(inplace=True),
)
```

Input shape `(B, 128*4*4) = (B, 2048)`; output shape `(B, 64, 8, 8)`. The `Rearrange` keeps the whole network expressible as one `nn.Sequential` — no custom `forward` needed to do the `.view(B, C, H, W)`.

### Composite Exercise — Rearrange unflattens the latent into a feature map before a ConvT/BN/ReLU block

**Atoms exercised together**: `convtranspose-bn-activation-block`, `rearrange-as-sequential-layer`

Implement `cx2_build_rearrange_then_g_block(latent_channels=128, h=4, w=4, out_channels=64)`. Return an `nn.Sequential` with exactly FOUR children, IN ORDER:

1. `Rearrange('b (c h w) -> b c h w', c=latent_channels, h=h, w=w)` (atom: rearrange-as-sequential-layer).
2. `nn.ConvTranspose2d(latent_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False)` (atom: convtranspose-bn-activation-block, layer 1/3).
3. `nn.BatchNorm2d(out_channels)` (atom: convtranspose-bn-activation-block, layer 2/3).
4. `nn.ReLU(inplace=True)` (atom: convtranspose-bn-activation-block, layer 3/3).

The test checks:
- Returned value is `nn.Sequential` with exactly 4 children.
- Child 0 is an `einops.layers.torch.Rearrange` (NOT a lambda module — must be the real einops layer).
- Children 1/2/3 are `ConvTranspose2d`, `BatchNorm2d`, `ReLU`.
- `ConvT.bias is None` (bias=False).
- Forward pass: input `(B, latent_channels*h*w)` produces `(B, out_channels, 2*h, 2*w)`.
- Output is non-negative.
- Numerically agrees with the manual two-step `(rearrange + conv block)` reference.

In [ ]:
def cx2_build_rearrange_then_g_block(latent_channels=128, h=4, w=4, out_channels=64):
    return nn.Sequential(
        # Atom A (rearrange-as-sequential-layer): einops layer-module form.
        Rearrange('b (c h w) -> b c h w', c=latent_channels, h=h, w=w),
        # Atom B (convtranspose-bn-activation-block).
        nn.ConvTranspose2d(latent_channels, out_channels,
                           kernel_size=4, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


<details><summary>Show solution — cx2</summary>

```python
def cx2_build_rearrange_then_g_block(latent_channels=128, h=4, w=4, out_channels=64):
    return nn.Sequential(
        # Atom A (rearrange-as-sequential-layer): einops layer-module form.
        Rearrange('b (c h w) -> b c h w', c=latent_channels, h=h, w=w),
        # Atom B (convtranspose-bn-activation-block).
        nn.ConvTranspose2d(latent_channels, out_channels,
                           kernel_size=4, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )
```

The `Rearrange` MODULE (`einops.layers.torch.Rearrange`) is the key to staying inside `nn.Sequential`. The plain `einops.rearrange` FUNCTION can't be inserted into a Sequential (it's not a Module). Common bug: shipping `c=latent_channels` as a default in the pattern string instead of as a kwarg — the kwarg form is what lets einops verify shape at runtime and produce a clear error if the latent length is wrong.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx2',
        'subtopics': ["GAN: ConvT+BN+Activation block", "Einops: Rearrange as nn.Sequential layer"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()